## EcoHome Energy Advisor - Agent Run & Evaluation

In this notebook, you'll run the Energy Advisor agent with various real-world scenarios and see how it helps customers optimize their energy usage.

## Learning Objectives
- Create the agent's instructions
- Run the Energy Advisor with different types of questions
- Evaluate response quality and accuracy
- Measure tool usage effectiveness
- Identify areas for improvement
- Implement evaluation metrics

## Evaluation Criteria
- **Accuracy**: Correct information and calculations
- **Relevance**: Responses address the user's question
- **Completeness**: Comprehensive answers with actionable advice
- **Tool Usage**: Appropriate use of available tools
- **Reasoning**: Clear explanation of recommendations


## 1. Import and Initialize

In [1]:
from datetime import datetime
from agent import Agent

In [2]:
## TODO: Create the agent's instructions

TODAY = datetime.now()

ECOHOME_SYSTEM_PROMPT = f"""
You are the **EcoHome Energy Advisor**, an expert AI assistant for EcoHome, a smart-home energy
management company. You help customers who own solar panels, electric vehicles (EVs) and smart
thermostats/appliances reduce their electricity costs and environmental impact.

## Current date
Today is {TODAY.strftime("%A, %Y-%m-%d")}. Use it to resolve relative dates such as "today",
"tomorrow", "this weekend" or "last week". Always pass dates to tools in YYYY-MM-DD format.

## Your goals
1. Minimize the customer's electricity cost.
2. Maximize self-consumption of on-site solar generation.
3. Reduce overall energy use and carbon footprint without sacrificing comfort or convenience.

## Available tools and when to use them
- `get_weather_forecast(location, days)`: weather, temperature and hourly solar irradiance.
  Use for anything involving solar production, heating/cooling, or planning future days.
- `get_electricity_prices(date)`: hourly time-of-use rates for a date (peak vs. off-peak,
  demand charges). Use for any timing or cost question.
- `query_energy_usage(start_date, end_date, device_type)`: historical consumption and cost from
  the customer's database. Valid `device_type` values: "EV", "HVAC", "appliance" (omit for all).
- `query_solar_generation(start_date, end_date)`: historical solar generation from the
  customer's panels.
- `get_recent_energy_summary(hours)`: quick overview of recent consumption and generation.
- `search_energy_tips(query, max_results)`: retrieves energy-saving best practices from the
  EcoHome knowledge base. Use to ground general advice in documented guidance.
- `calculate_energy_savings(device_type, current_usage_kwh, optimized_usage_kwh, price_per_kwh,
  optimized_price_per_kwh)`: computes daily, monthly and annual kWh/USD savings and percentages.
  **This is the only way you may produce a savings figure.** Pass **daily** kWh values (convert
  weekly/monthly totals first) and real rates from `get_electricity_prices`, not the default.
  - Usage reduction (fewer kWh): `current_usage_kwh` > `optimized_usage_kwh`, same price.
  - Load shifting (same kWh, cheaper hours): `current_usage_kwh` == `optimized_usage_kwh`,
    `price_per_kwh` = peak rate, `optimized_price_per_kwh` = off-peak rate.

## How to work
1. **Understand the request.** Identify the device(s), time frame and objective (cost, solar,
   comfort). Use the location from the provided context; if none is given, ask for it only when
   it is essential (e.g., weather), otherwise state a reasonable assumption.
2. **Gather data before advising.** Never guess prices, forecasts, or usage numbers—call the
   relevant tools. Typical combinations:
   - EV charging / appliance scheduling -> `get_electricity_prices` + `get_weather_forecast`
     (and `query_energy_usage` for typical load).
   - Thermostat / HVAC -> `get_weather_forecast` + `get_electricity_prices` + `search_energy_tips`.
   - Solar maximization -> `get_weather_forecast` + `query_solar_generation`.
   - Usage analysis / "why is my bill high" -> `query_energy_usage` / `get_recent_energy_summary`.
   - Savings estimates (any question about how much the customer could save) -> gather real
     numbers first (`get_electricity_prices`, and `query_energy_usage` for the customer's own
     load), then **always** call `calculate_energy_savings`.
   - General tips -> `search_energy_tips`.
   Call independent tools in parallel when possible, and do not call tools that are irrelevant.
3. **Reason over the data.**
   - Compare hourly rates: off-peak hours are cheaper (peak is roughly 06:00-22:00).
   - Solar output is highest around midday on sunny days and drops sharply when cloudy/rainy.
   - The best window balances low grid price with high solar availability. If strong solar
     overlaps peak hours, explain that self-generated solar is effectively free and can beat
     off-peak grid power; if solar is weak, prefer off-peak hours.
   - For thermostats, use pre-heating/pre-cooling before peak periods and setbacks when away
     or asleep; respect comfort (typically 68-72°F / 20-22°C in winter, 74-78°F / 23-26°C in
     summer).
4. **Quantify.** Whenever possible, give concrete numbers: kWh, $/kWh, estimated cost, and
   daily/monthly/annual savings. **Never compute savings yourself**: every savings number
   (kWh, $, %, daily/monthly/annual) must come from a `calculate_energy_savings` call. Show the
   inputs you passed (kWh, rates) so the result can be verified. If rates vary (e.g. a range of
   off-peak prices), call the tool once per scenario (e.g. low and high) to report a range.

## Example questions and how to approach them
- "When should I charge my electric car tomorrow to minimize cost and maximize solar power?"
  -> `get_weather_forecast` + `get_electricity_prices` for tomorrow; compare the solar window
  with off-peak hours and recommend specific charging times with cost estimates.
- "What temperature should I set my thermostat on Wednesday afternoon if electricity prices spike?"
  -> `get_electricity_prices` for Wednesday + `get_weather_forecast` (+ `search_energy_tips`);
  suggest pre-cooling/pre-heating before the spike and a setpoint during peak hours.
- "Suggest three ways I can reduce energy use based on my usage history."
  -> `query_energy_usage` (and/or `get_recent_energy_summary`) + `search_energy_tips`; give
  three concrete, data-backed actions targeting the largest consumers.
- "How much can I save by running my dishwasher during off-peak hours?"
  -> `get_electricity_prices` (+ `query_energy_usage` for appliance load) +
  `calculate_energy_savings` with the same daily kWh, `price_per_kwh` = peak rate and
  `optimized_price_per_kwh` = off-peak rate; show the peak vs. off-peak cost difference.
- "If I cut my pool pump from 10 kWh to 6 kWh per day, how much would I save?"
  -> `get_electricity_prices` + `calculate_energy_savings(current_usage_kwh=10,
  optimized_usage_kwh=6, price_per_kwh=<rate>)`; report the tool's daily, monthly and annual
  savings and percentage.
- "What's the best time this week to run my pool pump based on the weather forecast?"
  -> `get_weather_forecast` (days=7) + `get_electricity_prices`; recommend daily windows that
  align with strong solar output and/or off-peak rates.

## Response format
- If you retrieved a weather forecast for tomorrow (or any future date), open the response
  with a one-line weather summary stating the date and conditions, e.g.
  "**Tomorrow (<Weekday>, YYYY-MM-DD): <conditions>, <min>-<max>°C**". Then give the recommendation.
- Start with a short, direct answer (the recommendation in 1-2 sentences).
- Then provide:
  - **Recommended schedule / actions**: specific times (e.g., "11:00-14:00") or settings.
  - **Why**: the data that supports it (prices, forecast, solar, usage history).
  - **Estimated savings**: cost and energy figures with assumptions stated.
  - **Extra tips** (optional): 1-3 relevant, actionable tips.
- Use clear headings or bullet points; keep it concise and customer-friendly. Use the units
  returned by the tools (USD, kWh, °C) and convert to °F when helpful for US customers.

## Rules
- Be accurate: only cite numbers that come from tool outputs or clearly labeled estimates.
  Never fabricate data. Savings figures must come from `calculate_energy_savings`, never from
  your own arithmetic. If a tool returns an error or no data, say so and explain how that
  affects your recommendation.
- Be transparent about assumptions (e.g., EV battery size, charger power, typical usage).
- Stay within home energy topics. Politely decline unrelated requests.
- Prioritize safety: never advise disabling safety features, unsafe electrical work, or
  temperatures that could harm health, pets or the home. Suggest a licensed professional for
  electrical installation or repairs.
- Do not ask unnecessary follow-up questions; give the best possible answer with the data
  available, then optionally note what extra information would improve it.

"""


In [3]:
ecohome_agent = Agent(
    instructions=ECOHOME_SYSTEM_PROMPT,
)

In [4]:
response = ecohome_agent.invoke(
    question="When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
    context="Location: San Francisco, CA"
)

In [5]:
print(response["messages"][-1].content)

**Tomorrow (Wednesday, 2026-11-11): partly cloudy, 13.8–23.0°C**

Charge your EV **overnight from 22:00–06:00** if your goal is lowest cost, and use **11:00–16:00** if you want to maximize solar self-consumption. If you can split the session, a **midday top-up plus overnight finish** is the best balance.

### Recommended schedule / actions
- **Best for cost:** **22:00–06:00**
  - Off-peak rates are about **$0.097–$0.103/kWh** overnight.
- **Best for solar:** **11:00–16:00**
  - Solar irradiance is strongest then, peaking around **12:00 at 480** and still solid through **16:00**.
- **Best compromise:**  
  - Charge **11:00–14:00** with solar first, then finish any remaining charging **after 22:00**.

### Why
- Tomorrow is **partly cloudy**, so solar is available but weaker than a clear-sky day.
- Grid prices are much lower overnight:
  - **Off-peak:** 00:00–05:00 and 22:00–23:00
  - **Peak:** 06:00–21:00, roughly **$0.147–$0.154/kWh**
- Midday solar is the best window to use your own ge

In [6]:
print("TOOLS:")
for msg in response["messages"]:
    obj = msg.model_dump()
    if obj.get("tool_call_id"):
        print("-", msg.name)

TOOLS:
- get_weather_forecast
- get_electricity_prices


## 2. Define Test Cases

In [7]:
# TODO: Define comprehensive test cases for the Energy Advisor
# Create 10 test cases covering different scenarios:
# - EV charging optimization
# - Thermostat settings
# - Appliance scheduling
# - Solar power maximization
# - Cost savings calculations

In [8]:
test_cases = [
    # --- EV charging optimization ---
    {
        "id": "ev_charging_1",
        "question": "When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "The response should contain time recommendation, cost analysis and solar consideration",
    },
    {
        "id": "ev_charging_2",
        "question": "How much electricity did my EV use over the past week, and how much could I save by shifting charging to off-peak hours?",
        "expected_tools": ["query_energy_usage", "get_electricity_prices", "calculate_energy_savings"],
        "expected_response": "The response should report past EV consumption, identify off-peak hours, and quantify savings in kWh and dollars",
    },
    # --- Thermostat settings ---
    {
        "id": "thermostat_1",
        "question": "What temperature should I set my thermostat to tomorrow given the weather forecast and electricity prices?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "The response should give specific temperature setpoints by time of day, reference forecast temperatures, and consider peak pricing (e.g., pre-cooling/pre-heating)",
    },
    {
        "id": "thermostat_2",
        "question": "My HVAC system seems to use a lot of energy. What are some best practices for thermostat settings to cut heating and cooling costs?",
        "expected_tools": ["query_energy_usage", "search_energy_tips"],
        "expected_response": "The response should reference the user's HVAC usage and provide actionable thermostat tips (setpoints, setbacks, smart scheduling) drawn from the knowledge base",
    },
    # --- Appliance scheduling ---
    {
        "id": "appliance_scheduling_1",
        "question": "When is the best time to run my dishwasher and washing machine tomorrow?",
        "expected_tools": ["get_electricity_prices", "get_weather_forecast"],
        "expected_response": "The response should recommend specific time windows for each appliance based on low prices and/or solar generation, and explain the reasoning",
    },
    {
        "id": "appliance_scheduling_2",
        "question": "Which of my appliances used the most energy in the last 24 hours, and how should I reschedule them?",
        "expected_tools": ["get_recent_energy_summary", "get_electricity_prices"],
        "expected_response": "The response should rank devices by recent consumption and suggest rescheduling high-usage devices to cheaper hours",
    },
    # --- Solar power maximization ---
    {
        "id": "solar_maximization_1",
        "question": "How much solar power did I generate last week, and how can I use more of it directly instead of exporting to the grid?",
        "expected_tools": ["query_solar_generation", "query_energy_usage", "search_energy_tips"],
        "expected_response": "The response should summarize past solar generation, compare it with consumption, and give load-shifting tips for higher self-consumption",
    },
    {
        "id": "solar_maximization_2",
        "question": "Will the weather over the next 3 days be good for solar generation? Which day should I plan my high-energy tasks?",
        "expected_tools": ["get_weather_forecast"],
        "expected_response": "The response should assess cloud cover/irradiance per day, identify the best solar day(s), and recommend when to run high-energy tasks",
    },
    # --- Cost savings calculation ---
    {
        "id": "cost_savings_1",
        "question": "If I reduce my pool pump usage from 10 kWh to 6 kWh per day, how much would I save per day, month, and year?",
        "expected_tools": ["get_electricity_prices", "calculate_energy_savings"],
        "expected_response": "The response should compute savings of 4 kWh/day and show daily, monthly, and annual dollar savings with the percentage reduction",
    },
    {
        "id": "cost_savings_2",
        "question": "Based on my energy usage over the last month, what is my estimated electricity cost and what are the top 3 ways to reduce it?",
        "expected_tools": ["query_energy_usage", "get_electricity_prices", "search_energy_tips"],
        "expected_response": "The response should estimate monthly cost from actual usage and prices, and provide three prioritized, quantified savings recommendations",
    },
    # --- General / multi-topic ---
    {
        "id": "energy_summary_1",
        "question": "Give me a summary of my home's energy consumption for the last 48 hours.",
        "expected_tools": ["get_recent_energy_summary"],
        "expected_response": "The response should report total consumption, breakdown by device type, and highlight any unusual usage patterns",
    },
    {
        "id": "energy_tips_1",
        "question": "What are some tips for reducing standby (phantom) power consumption in my home?",
        "expected_tools": ["search_energy_tips"],
        "expected_response": "The response should give practical tips on phantom loads (smart power strips, unplugging devices, etc.) sourced from the knowledge base",
    },
]
print(f"Total test cases defined: {len(test_cases)}")
if len(test_cases) < 10:
    raise ValueError("You MUST have at least 10 test cases")


Total test cases defined: 12


## 3. Run Agent Tests

In [9]:
CONTEXT = "Location: San Francisco, CA"

In [10]:
# Run the agent tests
# For each test case, call the agent and collect the response
# Store results for evaluation

print("=== Running Agent Tests ===")
test_results = []

for i, test_case in enumerate(test_cases):
    print(f"\nTest {i+1}: {test_case['id']}")
    print(f"Question: {test_case['question']}")
    print("-" * 50)
    
    try:
        # Call the agent
        response = ecohome_agent.invoke(
            question=test_case['question'],
            context=CONTEXT
        )
        
        # Store the result for evaluation
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': response,
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat()
        }
        test_results.append(result)
                
    except Exception as e:
        print(f"Error: {e}")
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': f"Error: {str(e)}",
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat(),
            'error': str(e)
        }
        test_results.append(result)

print(f"\nCompleted {len(test_results)} tests")


=== Running Agent Tests ===

Test 1: ev_charging_1
Question: When should I charge my electric car tomorrow to minimize cost and maximize solar power?
--------------------------------------------------

Test 2: ev_charging_2
Question: How much electricity did my EV use over the past week, and how much could I save by shifting charging to off-peak hours?
--------------------------------------------------

Test 3: thermostat_1
Question: What temperature should I set my thermostat to tomorrow given the weather forecast and electricity prices?
--------------------------------------------------

Test 4: thermostat_2
Question: My HVAC system seems to use a lot of energy. What are some best practices for thermostat settings to cut heating and cooling costs?
--------------------------------------------------

Test 5: appliance_scheduling_1
Question: When is the best time to run my dishwasher and washing machine tomorrow?
--------------------------------------------------

Test 6: appliance_sche

In [11]:
test_results

[{'test_id': 'ev_charging_1',
  'question': 'When should I charge my electric car tomorrow to minimize cost and maximize solar power?',
  'response': {'messages': [SystemMessage(content='Location: San Francisco, CA', additional_kwargs={}, response_metadata={}, id='7fa36dfa-cc20-4416-abbd-06ca25dc2544'),
    HumanMessage(content='When should I charge my electric car tomorrow to minimize cost and maximize solar power?', additional_kwargs={}, response_metadata={}, id='a3ca2205-786d-4c7c-8ed0-61b3967c380e'),
    AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_RaJjLhRkbAzON809xx3c6gNL', 'function': {'arguments': '{"location": "San Francisco, CA", "days": 2}', 'name': 'get_weather_forecast'}, 'type': 'function'}, {'id': 'call_bUpMrnghpRqereXYg5jyOak0', 'function': {'arguments': '{"date": "2026-11-11"}', 'name': 'get_electricity_prices'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 3134, 'total_token

## 4. Evaluate Responses

In [12]:
# TODO: Implement evaluation functions
# Create functions to evaluate:
# - Final Response
# - Tool usage

In [13]:
# Response evaluator (LLM-as-a-judge)
import os
from typing import List

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

load_dotenv()

RESPONSE_METRICS = ["accuracy", "relevance", "completeness", "usefulness"]


class ResponseEvaluation(BaseModel):
    """Structured judgement of a single agent response (scores are 1-5)."""
    accuracy: int = Field(..., ge=1, le=5, description="Correct, internally consistent facts and calculations; no fabricated data")
    relevance: int = Field(..., ge=1, le=5, description="Directly addresses the user's question without off-topic content")
    completeness: int = Field(..., ge=1, le=5, description="Covers every element listed in the expected response")
    usefulness: int = Field(..., ge=1, le=5, description="Actionable, specific (times, setpoints, $/kWh) and customer-friendly")
    strengths: List[str] = Field(default_factory=list, description="What the response did well")
    weaknesses: List[str] = Field(default_factory=list, description="Gaps, errors or unclear parts")
    missing_elements: List[str] = Field(default_factory=list, description="Items from the expected response that are absent")
    feedback: str = Field(..., description="Comprehensive feedback paragraph with concrete improvement suggestions")


JUDGE_SYSTEM_PROMPT = """
You are a strict, impartial evaluator of an AI home-energy advisor (EcoHome Energy Advisor).
Given a customer QUESTION, the advisor's FINAL RESPONSE and a description of the EXPECTED RESPONSE,
score the final response on a 1-5 integer scale for each metric:

- ACCURACY: Are facts, numbers and calculations correct and internally consistent
  (e.g., kWh x $/kWh = cost, daily -> monthly -> annual conversions)? Are claims plausible and
  not fabricated? Penalize math errors and contradictions.
- RELEVANCE: Does the response directly answer the question asked (device, date, goal)?
  Penalize off-topic or generic filler.
- COMPLETENESS: Does it cover every element described in the EXPECTED RESPONSE?
  Penalize each missing element.
- USEFULNESS: Is the advice actionable and specific (time windows, setpoints, quantified savings,
  stated assumptions) and easy for a customer to follow?

Scale: 5 = excellent, 4 = good with minor issues, 3 = acceptable but notable gaps,
2 = poor, 1 = fails / wrong / no answer.
Then list strengths, weaknesses, missing elements, and write comprehensive feedback with
concrete suggestions for improvement.
""".strip()

_judge_llm = ChatOpenAI(
    model="gpt-5.4",  # larger model than gpt-5.4-mini
    temperature=0.0,
    base_url="https://openai.vocareum.com/v1",
    api_key=os.getenv("VOCAREUM_API_KEY"),
).with_structured_output(ResponseEvaluation)


def evaluate_response(question, final_response, expected_response):
    """Evaluate a single response against expected response.

    Returns a dict with per-metric scores (1-5), normalized scores (0-1),
    an overall score, and comprehensive feedback.
    """
    # Errors / empty responses cannot be judged: give the minimum score.
    if not final_response or str(final_response).startswith("Error:"):
        scores = {m: 1 for m in RESPONSE_METRICS}
        return {
            "scores": scores,
            "normalized_scores": {m: 0.0 for m in RESPONSE_METRICS},
            "overall_score": 0.0,
            "strengths": [],
            "weaknesses": ["Agent returned no usable response."],
            "missing_elements": [expected_response],
            "feedback": f"The agent failed to produce a response: {final_response}",
        }

    user_prompt = (
        f"QUESTION:\n{question}\n\n"
        f"EXPECTED RESPONSE (criteria):\n{expected_response}\n\n"
        f"FINAL RESPONSE:\n{final_response}"
    )

    try:
        judgement: ResponseEvaluation = _judge_llm.invoke([
            SystemMessage(content=JUDGE_SYSTEM_PROMPT),
            HumanMessage(content=user_prompt),
        ])
    except Exception as e:
        return {
            "scores": {m: None for m in RESPONSE_METRICS},
            "normalized_scores": {m: None for m in RESPONSE_METRICS},
            "overall_score": None,
            "strengths": [],
            "weaknesses": [],
            "missing_elements": [],
            "feedback": f"Evaluation failed: {e}",
        }

    scores = {m: getattr(judgement, m) for m in RESPONSE_METRICS}
    normalized = {m: (s - 1) / 4 for m, s in scores.items()}  # map 1-5 -> 0-1
    overall_score = round(sum(normalized.values()) / len(normalized), 3)

    return {
        "scores": scores,
        "normalized_scores": normalized,
        "overall_score": overall_score,
        "strengths": judgement.strengths,
        "weaknesses": judgement.weaknesses,
        "missing_elements": judgement.missing_elements,
        "feedback": judgement.feedback,
    }


# Quick check on the first test result
_r = test_results[0]
_eval = evaluate_response(
    question=_r["question"],
    final_response=_r["response"]["messages"][-1].content if isinstance(_r["response"], dict) else _r["response"],
    expected_response=_r["expected_response"],
)
print("Scores:", _eval["scores"], "| Overall:", _eval["overall_score"])
print("Feedback:", _eval["feedback"])

Scores: {'accuracy': 4, 'relevance': 5, 'completeness': 5, 'usefulness': 5} | Overall: 0.938
Feedback: This is a strong response that directly answers the customer's question with specific time windows, relevant price information, and practical guidance on balancing cost versus solar usage. It fully covers the expected elements: time recommendation, cost analysis, and solar consideration. To improve further, make the solar data clearer by using standard units (for example W/m² or estimated solar generation) and present the peak range in a sensible order. It would also be more useful to quantify the savings for a sample EV charge, such as comparing the cost of charging 10 kWh overnight versus during peak-rate hours, and, if possible, estimate how much of the midday charging could be covered by solar under partly cloudy conditions.


In [14]:
# Tool usage evaluator (deterministic, based on the agent's message trace)
from collections import Counter

from langchain_core.messages import AIMessage, ToolMessage


def _extract_tool_trace(messages):
    """Return (ordered list of called tool names, list of tool names whose call failed)."""
    called, failed = [], []
    for msg in messages or []:
        if isinstance(msg, AIMessage) and getattr(msg, "tool_calls", None):
            called.extend(tc["name"] for tc in msg.tool_calls)
        elif isinstance(msg, ToolMessage):
            content = str(msg.content)
            if getattr(msg, "status", None) == "error" or content.startswith("Error"):
                failed.append(msg.name)
    return called, failed


def evaluate_tool_usage(messages, expected_tools):
    """Evaluate if the right tools were used.

    Metrics (0-1):
      - tool_appropriateness: share of distinct tools called that were expected
        (precision -> penalizes irrelevant tools).
      - tool_completeness: share of expected tools that were actually called
        (recall -> penalizes missing tools).
      - tool_f1: harmonic mean of the two.
    Also reports redundant calls, failed calls, and comprehensive feedback.
    """
    expected = set(expected_tools or [])

    # The run itself errored -> no message trace to evaluate.
    if not isinstance(messages, list):
        return {
            "tools_called": [],
            "expected_tools": sorted(expected),
            "correct_tools": [],
            "missing_tools": sorted(expected),
            "unexpected_tools": [],
            "redundant_calls": {},
            "failed_tools": [],
            "tool_appropriateness": 0.0,
            "tool_completeness": 0.0,
            "tool_f1": 0.0,
            "feedback": "No message trace available (agent run failed), so no tools were used.",
        }

    called, failed = _extract_tool_trace(messages)
    used = set(called)

    # set([a, b, c]) & set([a, c]) returns the intersection, e.g., set(['a', 'c']).
    correct = used & expected
    # set([a, b, c]) - set([a, c]) returns the difference, e.g., set(['b']).
    missing = expected - used
    unexpected = used - expected
    redundant = {name: n for name, n in Counter(called).items() if n > 1}

    # Appropriateness (precision): if nothing was called, it is only "appropriate" when nothing was expected.
    appropriateness = len(correct) / len(used) if used else (1.0 if not expected else 0.0)
    # Completeness (recall)
    completeness = len(correct) / len(expected) if expected else 1.0
    f1 = (2 * appropriateness * completeness / (appropriateness + completeness)
          if (appropriateness + completeness) else 0.0)

    # --- Comprehensive feedback ---
    fb = []
    if not used:
        fb.append("The agent answered without calling any tools"
                  + (f", although {sorted(expected)} were expected; the answer is likely not grounded in real data."
                     if expected else ", which is acceptable for this question."))
    else:
        fb.append(f"Called {len(called)} tool(s) ({len(used)} distinct): {called}.")
        if correct:
            fb.append(f"Correctly used: {sorted(correct)}.")
        if missing:
            fb.append(f"Missing required tool(s): {sorted(missing)} -> the response may rely on assumptions "
                      "instead of data; strengthen the system prompt's tool-selection guidance for this scenario.")
        else:
            fb.append("All required tools were used.")
        if unexpected:
            fb.append(f"Extra tool(s) not in the expected set: {sorted(unexpected)}. "
                      "Check whether they added useful context or were unnecessary (extra latency/cost).")
        if redundant:
            fb.append(f"Repeated calls: {redundant}. Consider whether these were needed "
                      "(e.g., different dates) or wasted calls.")
        if failed:
            fb.append(f"Tool call(s) returned errors: {sorted(set(failed))}. Verify arguments "
                      "(date format, device_type values) and that the response acknowledges missing data.")

    if f1 == 1.0 and not failed:
        verdict = "Excellent tool usage."
    elif completeness >= 0.5 and appropriateness >= 0.5:
        verdict = "Acceptable tool usage with room for improvement."
    else:
        verdict = "Poor tool usage."
    fb.insert(0, verdict)

    return {
        "tools_called": called,
        "expected_tools": sorted(expected),
        "correct_tools": sorted(correct),
        "missing_tools": sorted(missing),
        "unexpected_tools": sorted(unexpected),
        "redundant_calls": redundant,
        "failed_tools": sorted(set(failed)),
        "tool_appropriateness": round(appropriateness, 3),
        "tool_completeness": round(completeness, 3),
        "tool_f1": round(f1, 3),
        "feedback": " ".join(fb),
    }


# Quick check on all test results
for r in test_results:
    msgs = r["response"]["messages"] if isinstance(r["response"], dict) else r["response"]
    t = evaluate_tool_usage(msgs, r["expected_tools"])
    print(f"{r['test_id']:<25} appropriateness={t['tool_appropriateness']:.2f} "
          f"completeness={t['tool_completeness']:.2f} f1={t['tool_f1']:.2f}")
    print(f"  {t['feedback']}")

ev_charging_1             appropriateness=1.00 completeness=1.00 f1=1.00
  Excellent tool usage. Called 2 tool(s) (2 distinct): ['get_weather_forecast', 'get_electricity_prices']. Correctly used: ['get_electricity_prices', 'get_weather_forecast']. All required tools were used.
ev_charging_2             appropriateness=1.00 completeness=1.00 f1=1.00
  Excellent tool usage. Called 3 tool(s) (3 distinct): ['query_energy_usage', 'get_electricity_prices', 'calculate_energy_savings']. Correctly used: ['calculate_energy_savings', 'get_electricity_prices', 'query_energy_usage']. All required tools were used.
thermostat_1              appropriateness=0.67 completeness=1.00 f1=0.80
  Acceptable tool usage with room for improvement. Called 3 tool(s) (3 distinct): ['get_weather_forecast', 'get_electricity_prices', 'search_energy_tips']. Correctly used: ['get_electricity_prices', 'get_weather_forecast']. All required tools were used. Extra tool(s) not in the expected set: ['search_energy_tips']. Ch

In [15]:
# Comprehensive evaluation report
# - Calculate overall scores and metrics
# - Identify strengths and weaknesses
# - Provide recommendations for improvement
from collections import Counter, defaultdict
from statistics import mean

# Weight of each part in the combined per-test score (0-1)
RESPONSE_WEIGHT, TOOL_WEIGHT = 0.6, 0.4
PASS_THRESHOLD = 0.7          # combined score needed to "pass" a test
STRONG_METRIC, WEAK_METRIC = 4.0, 3.5   # thresholds on the 1-5 response scale
STRONG_TOOL, WEAK_TOOL = 0.9, 0.8       # thresholds on the 0-1 tool scale

METRIC_ADVICE = {
    "accuracy": "Require the agent to cite only tool outputs and to show key calculations "
                "(kWh x $/kWh, daily -> monthly -> annual) so numbers can be verified.",
    "relevance": "Tighten the system prompt so the agent answers the exact device/date/goal first "
                 "and drops generic filler.",
    "completeness": "Add a checklist to the response format (schedule, data rationale, savings, assumptions) "
                    "and make sure every part of multi-part questions is answered.",
    "usefulness": "Ask for concrete time windows, setpoints and quantified savings with stated "
                  "assumptions instead of general advice.",
}


def _safe_mean(values):
    values = [v for v in values if v is not None]
    return round(mean(values), 3) if values else None


def _final_text(response):
    if isinstance(response, dict) and response.get("messages"):
        return response["messages"][-1].content
    return str(response)


def generate_evaluation_report(results=None):
    """Evaluate every test result and build a structured report (dict).

    Returns:
        {
          "generated_at", "num_tests", "num_errors",
          "summary":     overall/aggregate scores and pass rate,
          "by_category": average scores per scenario category,
          "per_test":    detailed results for each test,
          "strengths", "weaknesses", "recommendations": lists of strings,
        }
    """
    results = test_results if results is None else results

    # ---------- 1. Evaluate each test ----------
    per_test = []
    for r in results:
        messages = r["response"]["messages"] if isinstance(r["response"], dict) else r["response"]
        resp_eval = evaluate_response(r["question"], _final_text(r["response"]), r["expected_response"])
        tool_eval = evaluate_tool_usage(messages, r["expected_tools"])

        resp_score = resp_eval["overall_score"]
        combined = (round(RESPONSE_WEIGHT * resp_score + TOOL_WEIGHT * tool_eval["tool_f1"], 3)
                    if resp_score is not None else None)

        per_test.append({
            "test_id": r["test_id"],
            "category": r["test_id"].rsplit("_", 1)[0],
            "question": r["question"],
            "error": r.get("error"),
            "response_eval": resp_eval,
            "tool_eval": tool_eval,
            "combined_score": combined,
            "passed": combined is not None and combined >= PASS_THRESHOLD,
        })

    # ---------- 2. Aggregate metrics ----------
    metric_avgs = {m: _safe_mean([t["response_eval"]["scores"][m] for t in per_test])
                   for m in RESPONSE_METRICS}
    tool_avgs = {k: _safe_mean([t["tool_eval"][k] for t in per_test])
                 for k in ("tool_appropriateness", "tool_completeness", "tool_f1")}
    scored = [t for t in per_test if t["combined_score"] is not None]

    summary = {
        "overall_score": _safe_mean([t["combined_score"] for t in per_test]),
        "response_score": _safe_mean([t["response_eval"]["overall_score"] for t in per_test]),
        "response_metrics_1to5": metric_avgs,
        "tool_metrics": tool_avgs,
        "pass_rate": round(sum(t["passed"] for t in per_test) / len(per_test), 3) if per_test else 0.0,
        "num_passed": sum(t["passed"] for t in per_test),
        "num_evaluation_failures": len(per_test) - len(scored),
    }

    by_category = defaultdict(list)
    for t in per_test:
        by_category[t["category"]].append(t)
    by_category = {
        cat: {
            "num_tests": len(ts),
            "combined_score": _safe_mean([t["combined_score"] for t in ts]),
            "response_score": _safe_mean([t["response_eval"]["overall_score"] for t in ts]),
            "tool_f1": _safe_mean([t["tool_eval"]["tool_f1"] for t in ts]),
        }
        for cat, ts in by_category.items()
    }

    missing_counter = Counter(tool for t in per_test for tool in t["tool_eval"]["missing_tools"])
    unexpected_counter = Counter(tool for t in per_test for tool in t["tool_eval"]["unexpected_tools"])
    failed_counter = Counter(tool for t in per_test for tool in t["tool_eval"]["failed_tools"])
    ranked = sorted(scored, key=lambda t: t["combined_score"], reverse=True)

    # ---------- 3. Strengths ----------
    strengths = []
    for m, avg in metric_avgs.items():
        if avg is not None and avg >= STRONG_METRIC:
            strengths.append(f"High {m.upper()} (avg {avg:.2f}/5).")
    if tool_avgs["tool_appropriateness"] is not None and tool_avgs["tool_appropriateness"] >= STRONG_TOOL:
        strengths.append(f"Selects appropriate tools (appropriateness {tool_avgs['tool_appropriateness']:.2f}).")
    if tool_avgs["tool_completeness"] is not None and tool_avgs["tool_completeness"] >= STRONG_TOOL:
        strengths.append(f"Uses all required tools (completeness {tool_avgs['tool_completeness']:.2f}).")
    for cat, s in by_category.items():
        if s["combined_score"] is not None and s["combined_score"] >= 0.85:
            strengths.append(f"Strong on '{cat}' scenarios (combined {s['combined_score']:.2f}).")
    for t in ranked[:2]:
        top = t["response_eval"]["strengths"][:1]
        strengths.append(f"Best test '{t['test_id']}' ({t['combined_score']:.2f})"
                         + (f": {top[0]}" if top else "."))

    # ---------- 4. Weaknesses ----------
    weaknesses = []
    for m, avg in metric_avgs.items():
        if avg is not None and avg < WEAK_METRIC:
            weaknesses.append(f"Low {m.upper()} (avg {avg:.2f}/5).")
    if tool_avgs["tool_appropriateness"] is not None and tool_avgs["tool_appropriateness"] < WEAK_TOOL:
        weaknesses.append(f"Calls unnecessary tools (appropriateness {tool_avgs['tool_appropriateness']:.2f}).")
    if tool_avgs["tool_completeness"] is not None and tool_avgs["tool_completeness"] < WEAK_TOOL:
        weaknesses.append(f"Skips required tools (completeness {tool_avgs['tool_completeness']:.2f}).")
    if missing_counter:
        weaknesses.append(f"Most frequently missed tools: {dict(missing_counter.most_common(3))}.")
    if failed_counter:
        weaknesses.append(f"Tool calls returned errors: {dict(failed_counter)}.")
    for t in per_test:
        if t["error"]:
            weaknesses.append(f"Agent run failed on '{t['test_id']}': {t['error']}")
        elif not t["passed"] and t["combined_score"] is not None:
            top = t["response_eval"]["weaknesses"][:1]
            weaknesses.append(f"Below threshold on '{t['test_id']}' ({t['combined_score']:.2f})"
                              + (f": {top[0]}" if top else "."))

    # ---------- 5. Recommendations ----------
    recommendations = []
    for m, avg in sorted(metric_avgs.items(), key=lambda kv: kv[1] if kv[1] is not None else 99):
        if avg is not None and avg < STRONG_METRIC:
            recommendations.append(f"[{m.upper()}] {METRIC_ADVICE[m]}")
    for tool, n in missing_counter.most_common():
        recommendations.append(f"[TOOLS] '{tool}' was missed in {n} test(s): add explicit guidance/examples "
                               f"in the system prompt for when to call it.")
    for tool, n in unexpected_counter.most_common(3):
        recommendations.append(f"[TOOLS] '{tool}' was called unexpectedly {n} time(s): review whether it adds "
                               f"value or update the expected tools / prompt to avoid unnecessary calls.")
    if failed_counter:
        recommendations.append("[ROBUSTNESS] Validate tool arguments (date format, device_type) and make sure "
                               "the agent explains how missing data affects its advice.")
    for cat, s in by_category.items():
        if s["combined_score"] is not None and s["combined_score"] < PASS_THRESHOLD:
            recommendations.append(f"[SCENARIO] Improve '{cat}' scenarios (combined {s['combined_score']:.2f}): "
                                   f"add a worked example for this case to the system prompt.")
    if not recommendations:
        recommendations.append("All metrics are above target. Add harder / edge-case tests "
                               "(missing data, ambiguous dates, off-topic requests) to keep improving.")

    return {
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "num_tests": len(per_test),
        "num_errors": sum(1 for t in per_test if t["error"]),
        "summary": summary,
        "by_category": by_category,
        "per_test": per_test,
        "strengths": strengths,
        "weaknesses": weaknesses,
        "recommendations": recommendations,
    }

In [16]:
# Display the evaluation report (kept separate from report generation)
def _fmt(v, spec=".2f"):
    return "n/a" if v is None else format(v, spec)


def display_evaluation_report(report, show_feedback=True):
    """Pretty-print a report produced by generate_evaluation_report()."""
    line = "=" * 80
    s = report["summary"]

    print(line)
    print("ECOHOME ENERGY ADVISOR - EVALUATION REPORT".center(80))
    print(f"Generated: {report['generated_at']} | Tests: {report['num_tests']} | "
          f"Agent errors: {report['num_errors']}".center(80))
    print(line)

    # 1. Overall scores
    print("\n## 1. OVERALL SCORES")
    print(f"  Overall score (0-1):   {_fmt(s['overall_score'])}  "
          f"(= {RESPONSE_WEIGHT:.0%} response + {TOOL_WEIGHT:.0%} tool F1)")
    print(f"  Response score (0-1):  {_fmt(s['response_score'])}")
    print(f"  Pass rate:             {s['num_passed']}/{report['num_tests']} "
          f"({_fmt(s['pass_rate'], '.0%')}, threshold {PASS_THRESHOLD})")
    if s["num_evaluation_failures"]:
        print(f"  Evaluation failures:   {s['num_evaluation_failures']}")

    print("\n  Response metrics (avg, 1-5):")
    for m, v in s["response_metrics_1to5"].items():
        bar = "#" * int(round((v or 0) * 4))
        print(f"    {m.upper():<14} {_fmt(v):>5}  {bar}")
    print("\n  Tool metrics (avg, 0-1):")
    for k, v in s["tool_metrics"].items():
        bar = "#" * int(round((v or 0) * 20))
        print(f"    {k:<22} {_fmt(v):>5}  {bar}")

    # 2. By category
    print("\n## 2. SCORES BY CATEGORY")
    print(f"  {'Category':<24}{'Tests':>6}{'Combined':>10}{'Response':>10}{'Tool F1':>9}")
    for cat, c in sorted(report["by_category"].items(), key=lambda kv: kv[1]["combined_score"] or 0):
        print(f"  {cat:<24}{c['num_tests']:>6}{_fmt(c['combined_score']):>10}"
              f"{_fmt(c['response_score']):>10}{_fmt(c['tool_f1']):>9}")

    # 3. Per test
    print("\n## 3. PER-TEST RESULTS")
    print(f"  {'Test':<24}{'Acc':>4}{'Rel':>4}{'Comp':>5}{'Use':>4}{'ToolApp':>8}{'ToolComp':>9}"
          f"{'Combined':>9}  Result")
    for t in report["per_test"]:
        sc, te = t["response_eval"]["scores"], t["tool_eval"]
        print(f"  {t['test_id']:<24}{_fmt(sc['accuracy'], 'd') if sc['accuracy'] else 'n/a':>4}"
              f"{_fmt(sc['relevance'], 'd') if sc['relevance'] else 'n/a':>4}"
              f"{_fmt(sc['completeness'], 'd') if sc['completeness'] else 'n/a':>5}"
              f"{_fmt(sc['usefulness'], 'd') if sc['usefulness'] else 'n/a':>4}"
              f"{te['tool_appropriateness']:>8.2f}{te['tool_completeness']:>9.2f}"
              f"{_fmt(t['combined_score']):>9}  {'PASS' if t['passed'] else 'FAIL'}")
        if te["missing_tools"]:
            print(f"      missing tools: {te['missing_tools']}")
        if te["unexpected_tools"]:
            print(f"      extra tools:   {te['unexpected_tools']}")

    # 4-6. Strengths / weaknesses / recommendations
    for title, items in (("4. STRENGTHS", report["strengths"]),
                         ("5. WEAKNESSES", report["weaknesses"]),
                         ("6. RECOMMENDATIONS", report["recommendations"])):
        print(f"\n## {title}")
        for i, item in enumerate(items or ["(none)"], 1):
            print(f"  {i}. {item}")

    # 7. Detailed feedback
    if show_feedback:
        print("\n## 7. DETAILED FEEDBACK")
        for t in report["per_test"]:
            print(f"\n  [{t['test_id']}] {t['question']}")
            print(f"    Response: {t['response_eval']['feedback']}")
            print(f"    Tools:    {t['tool_eval']['feedback']}")
    print("\n" + line)


evaluation_report = generate_evaluation_report(test_results)
display_evaluation_report(evaluation_report)

                   ECOHOME ENERGY ADVISOR - EVALUATION REPORT                   
          Generated: 2026-11-10T13:45:17 | Tests: 12 | Agent errors: 0          

## 1. OVERALL SCORES
  Overall score (0-1):   0.78  (= 60% response + 40% tool F1)
  Response score (0-1):  0.77
  Pass rate:             8/12 (67%, threshold 0.7)

  Response metrics (avg, 1-5):
    ACCURACY        3.83  ###############
    RELEVANCE       4.58  ##################
    COMPLETENESS    3.83  ###############
    USEFULNESS      4.08  ################

  Tool metrics (avg, 0-1):
    tool_appropriateness    0.79  ################
    tool_completeness       0.81  ################
    tool_f1                 0.79  ################

## 2. SCORES BY CATEGORY
  Category                 Tests  Combined  Response  Tool F1
  thermostat                   2      0.69      0.75     0.60
  cost_savings                 2      0.69      0.69     0.70
  solar_maximization           2      0.71      0.72     0.70
  appliance_sc